# Ensemble Model Testing

This notebook tests the K-fold ensemble loading functionality.

## Features Tested:
- Loading ensemble model from K-fold checkpoints
- Weighted voting based on fold performance (F1 scores)
- Automatic checkpoint search across multiple locations
- Performance validation (95% threshold)
- Automatic retraining for missing/degraded checkpoints

In [1]:
from GradientGang.Pipeline.FinalPipeline import FinalPipeline
import torch
import os
%load_ext autoreload
%autoreload 2

## Initialize FinalPipeline

Set up the pipeline with your project configuration.

In [2]:
# Initialize FinalPipeline with parameters
import os
import dotenv

dotenv.load_dotenv()
database_url = os.getenv("DATABASE_URL")

data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'train_global_features_file': "train_global_features.csv",
    'test_global_features_file': "test_global_features.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.2,
    'shuffle': True,
    'use_kfold': True,
    'n_folds': 4,  # Should match your training configuration
}

params = {
    "database_url": database_url,
    "data_params": data_params,
    'project_name': 'TEST',
    'study_name': '_TEST6',
    'submission_path': '../Submissions/',
}

pipeline = FinalPipeline(params)

Storage: postgresql://postgres...
✓ Database configuration loaded
✓ Database initialized successfully!


[I 2025-11-14 19:11:57,432] Using an existing study with name 'TEST_TEST6' instead of creating a new one.


✓ Study created/loaded successfully!
Study name: TEST_TEST6
Sampler: TPESampler
Pruner: MedianPruner
Storage: Database
Total trials: 3
✓ Study initialized successfully!
✓ FinalPipeline initialized successfully!
Total trials: 3
✓ Study initialized successfully!
✓ FinalPipeline initialized successfully!


## Check Study Information

Verify that your Optuna study has completed trials with K-fold results.

In [3]:
# Display study summary
pipeline.study_summary()

# Get best trial
best_trial = pipeline.study.best_trial
print(f"\nBest trial number: {best_trial.number}")
print(f"Best F1 score: {best_trial.value:.4f}")

# Check if fold_scores exist
if 'fold_scores' in best_trial.user_attrs:
    fold_scores_str = best_trial.user_attrs['fold_scores']
    fold_scores = [float(x) for x in fold_scores_str.split(',')]
    print(f"\nFold scores: {fold_scores}")
    print(f"Number of folds: {len(fold_scores)}")
else:
    print("\n⚠️ WARNING: Best trial does not have 'fold_scores' attribute!")
    print("Make sure you ran training with K-fold cross-validation.")

Study name: TEST_TEST6
Direction: 2
Total trials: 3
Total trials: 3
Completed trials: 1
Completed trials: 1
Failed trials: 0
Failed trials: 0
Pruned trials: 1
Pruned trials: 1
Running trials: 1
Running trials: 1

✓ Best trial: 1
✓ Best F1 score: 0.9138

Top 5 trials:
  1. Trial 1: F1=0.9138 | Autoencoder | Conv1d

📊 Trial states (last 10):

✓ Best trial: 1
✓ Best F1 score: 0.9138

Top 5 trials:
  1. Trial 1: F1=0.9138 | Autoencoder | Conv1d

📊 Trial states (last 10):
  ⊗ Trial 0: PRUNED | N/A
  ✓ Trial 1: COMPLETE | F1=0.9138
  ⟳ Trial 2: RUNNING | N/A

Best trial number: 1
Best F1 score: 0.9138

Fold scores: [0.903614, 0.939394, 0.921212, 0.890909]
Number of folds: 4
  ⊗ Trial 0: PRUNED | N/A
  ✓ Trial 1: COMPLETE | F1=0.9138
  ⟳ Trial 2: RUNNING | N/A

Best trial number: 1
Best F1 score: 0.9138

Fold scores: [0.903614, 0.939394, 0.921212, 0.890909]
Number of folds: 4


In [4]:
# Check what parameters are actually stored in best trial
best_trial = pipeline.study.best_trial
print("Parameters in best trial:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")
    
print(f"\n🔍 Check if 'numFFLayers' is present: {'numFFLayers' in best_trial.params}")

Parameters in best trial:
  MacroArchitecture: Autoencoder
  use_windowing: False
  globalEmbeddingDim: 66
  globalNumLayers: 3
  globalDropout: 0.447413675213824
  globalActivation: LeakyReLU
  globalHiddenDim_0: 51
  globalHiddenDim_1: 36
  architectureType: Conv1d
  numConvLayers: 3
  kernelSize: 7
  strideConv1D: 2
  encoderActivation: LeakyReLU
  convChannels_0: 76
  poolType_0: avg
  convChannels_1: 191
  poolType_1: avg
  convChannels_2: 48
  poolType_2: max
  numFFLayers: 3
  ffHiddenDim: 172
  ffDropout: 0.165449012426325
  ffActivation: GELU
  ReconstructionLossWeight: 0.683684942670451
  LearningRate: 0.000817847657433954
  RegularizationWeight: 3.53875886477924
  SchedulerType: CosineAnnealingWarmRestarts
  T_0: 17
  T_mult: 2
  eta_min: 3.48284670652688e-06
  EarlyStoppingPatience: 15

🔍 Check if 'numFFLayers' is present: True


## Test Checkpoint Search

Check if checkpoints can be found for each fold.

In [5]:
best_trial = pipeline.study.best_trial
n_folds = data_params['n_folds']
studyName = params['project_name'] + params['study_name']

print(f"Searching for checkpoints for trial {best_trial.number} of study {studyName}...\n")

found_checkpoints = []
missing_folds = []

for fold_idx in range(n_folds):
    ckpt_path = pipeline._search_fold_checkpoint(
        trial_number=best_trial.number,
        fold_idx=fold_idx,
        study_name=studyName
    )
    
    if ckpt_path:
        print(f"✓ Fold {fold_idx}: Found checkpoint")
        print(f"  Path: {ckpt_path}")
        found_checkpoints.append(ckpt_path)
    else:
        print(f"✗ Fold {fold_idx}: Checkpoint NOT found (will retrain)")
        missing_folds.append(fold_idx)

print(f"\nSummary:")
print(f"  Found: {len(found_checkpoints)}/{n_folds}")
print(f"  Missing: {len(missing_folds)}/{n_folds}")

if missing_folds:
    print(f"\n⚠️ Folds {missing_folds} will be retrained automatically during ensemble loading.")

Searching for checkpoints for trial 1 of study TEST_TEST6...

✓ Fold 0: Found checkpoint
  Path: FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_0/version_0/checkpoints\trial-1-fold-0-epoch=33-val_F1=0.904.ckpt
✓ Fold 1: Found checkpoint
  Path: FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_1/version_0/checkpoints\trial-1-fold-1-epoch=40-val_F1=0.939.ckpt
✓ Fold 2: Found checkpoint
  Path: FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_2/version_0/checkpoints\trial-1-fold-2-epoch=36-val_F1=0.921.ckpt
✓ Fold 3: Found checkpoint
  Path: FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_3/version_0/checkpoints\trial-1-fold-3-epoch=24-val_F1=0.891.ckpt

Summary:
  Found: 4/4
  Missing: 0/4


## Load Ensemble Model

This will:
1. Search for checkpoints for each fold
2. Load and validate existing checkpoints
3. Retrain any missing or degraded folds
4. Create ensemble with weighted voting (weights = normalized F1 scores)

In [6]:
print("=" * 70)
print("LOADING ENSEMBLE WITH FORCE RETRAIN")
print("=" * 70)
print("\nThis will skip checkpoint validation and retrain all folds from")
print("scratch to ensure consistent architecture across all models.\n")

# Force retrain to ensure consistent architecture across all folds
ensemble = pipeline.load_ensemble_model(performance_threshold=0.95, force_retrain=False)

print("\n" + "=" * 70)
print("[SUCCESS] Ensemble created with consistent architecture!")
print("=" * 70)

LOADING ENSEMBLE WITH FORCE RETRAIN

This will skip checkpoint validation and retrain all folds from
scratch to ensure consistent architecture across all models.


LOADING K-FOLD ENSEMBLE
Best trial: 1
Best mean F1: 0.9138
Architecture: Autoencoder
------------------------------------------------------------
Best trial: 1
Best mean F1: 0.9138
Architecture: Autoencoder
------------------------------------------------------------
Reconstructing architecture...
✓ Architecture reconstructed

[STEP 1/3] Searching for checkpoints...
------------------------------------------------------------
  Fold 0: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_0/version_0/checkpoints\trial-1-fold-0-epoch=33-val_F1=0.904.ckpt
  Fold 1: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_1/version_0/checkpoints\trial-1-fold-1-epoch=40-val_F1=0.939.ckpt
  Fold 2: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_2/version_0/checkpoints\trial-1-fold-2-epoc

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


Validating fold 0 (expected F1: 0.9036)...
⚠️ Fold 0 degraded: F1=0.8157 < 0.8584
Fold 0: Validation failed - retraining...

🔄 Retraining fold 0...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...
   Training completed. Best F1: 0.8916
✓ Fold 0 retrained successfully: F1=0.8916
   Training completed. Best F1: 0.8916
✓ Fold 0 retrained successfully: F1=0.8916


GPU available: False, used: False


Validating fold 1 (expected F1: 0.9394)...
✓ Fold 1 validated: F1=0.8932
Validating fold 2 (expected F1: 0.9212)...
⚠️ Fold 2 degraded: F1=0.7962 < 0.8752
Fold 2: Validation failed - retraining...

🔄 Retraining fold 2...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...


TPU available: False, using: 0 TPU cores
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


   Training completed. Best F1: 0.8788
✓ Fold 2 retrained successfully: F1=0.8788
Validating fold 3 (expected F1: 0.8909)...
⚠️ Fold 3 degraded: F1=0.7578 < 0.8464
Fold 3: Validation failed - retraining...

🔄 Retraining fold 3...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...
   Training completed. Best F1: 0.8545
✓ Fold 3 retrained successfully: F1=0.8545
------------------------------------------------------------
[OK] All 4 models loaded/retrained
Fold F1 scores: ['0.8916', '0.8932', '0.8788', '0.8545']

✓ Ensemble created with performance-based weights
Weights: ['0.253', '0.254', '0.250', '0.243']
Device: cpu

[SUCCESS] Ensemble created with consistent architecture!
   Training completed. Best F1: 0.8545
✓ Fold 3 retrained successfully: F1=0.8545
------------------------------------------------------------
[OK] All 4 models loaded/retrained
Fold F1 scores: ['0.8916', 

In [12]:
print("=" * 70)
print("TEST: Architecture Validation (Before Loading Any Models)")
print("=" * 70)
print("\nThis test validates checkpoint consistency BEFORE attempting to")
print("load or retrain any models. It should detect the architecture")
print("mismatch in your existing checkpoints and fail with a clear error.\n")

try:
    # This should fail at the consistency check phase (STEP 2/3)
    ensemble = pipeline.load_ensemble_model(performance_threshold=0.95, force_retrain=False)
    print("\n[WARNING] Unexpected: Ensemble loaded without errors!")
    print("This means either:")
    print("  1. All checkpoints have consistent architecture, OR")
    print("  2. Only 0-1 checkpoints were found (consistency check skipped)")
except ValueError as e:
    print("\n" + "=" * 70)
    print("[SUCCESS] Architecture inconsistency detected as expected!")
    print("=" * 70)
    print(f"\nError details:\n{str(e)}")
    print("\n" + "=" * 70)
except Exception as e:
    print(f"\n[ERROR] Unexpected error type: {type(e).__name__}")
    print(f"Message: {str(e)}")

TEST: Architecture Validation (Before Loading Any Models)

This test validates checkpoint consistency BEFORE attempting to
load or retrain any models. It should detect the architecture
mismatch in your existing checkpoints and fail with a clear error.


LOADING K-FOLD ENSEMBLE
Best trial: 1
Best mean F1: 0.9138
Architecture: Autoencoder
------------------------------------------------------------
Best trial: 1
Best mean F1: 0.9138
Architecture: Autoencoder
------------------------------------------------------------
Reconstructing architecture...
✓ Architecture reconstructed

[STEP 1/3] Searching for checkpoints...
------------------------------------------------------------
  Fold 0: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_0/version_0/checkpoints\trial-1-fold-0-epoch=33-val_F1=0.904.ckpt
  Fold 1: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_1/version_0/checkpoints\trial-1-fold-1-epoch=40-val_F1=0.939.ckpt
  Fold 2: Found @ FinalPipelin

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


[OK] All 4 checkpoints have consistent architecture
  Architecture: {'encoder_layers': 6, 'feedforward_layers': 4, 'decoder_layers': 6, 'global_encoder_layers': 0, 'total_params': 34}

[STEP 3/3] Loading/retraining 4 fold models...
------------------------------------------------------------
Validating fold 0 (expected F1: 0.9036)...
⚠️ Fold 0 degraded: F1=0.8157 < 0.8584
Fold 0: Validation failed - retraining...

🔄 Retraining fold 0...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...
   Training completed. Best F1: 0.8193
✓ Fold 0 retrained successfully: F1=0.8193
Validating fold 1 (expected F1: 0.9394)...
✓ Fold 1 validated: F1=0.8932
   Training completed. Best F1: 0.8193
✓ Fold 0 retrained successfully: F1=0.8193
Validating fold 1 (expected F1: 0.9394)...
✓ Fold 1 validated: F1=0.8932


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


Validating fold 2 (expected F1: 0.9212)...
⚠️ Fold 2 degraded: F1=0.7962 < 0.8752
Fold 2: Validation failed - retraining...

🔄 Retraining fold 2...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


   Training completed. Best F1: 0.9030
✓ Fold 2 retrained successfully: F1=0.9030
Validating fold 3 (expected F1: 0.8909)...
⚠️ Fold 3 degraded: F1=0.7578 < 0.8464
Fold 3: Validation failed - retraining...

🔄 Retraining fold 3...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...
   Training completed. Best F1: 0.8848
✓ Fold 3 retrained successfully: F1=0.8848
------------------------------------------------------------
[OK] All 4 models loaded/retrained
Fold F1 scores: ['0.8193', '0.8932', '0.9030', '0.8848']

✓ Ensemble created with performance-based weights
Weights: ['0.234', '0.255', '0.258', '0.253']
Device: cpu

[WARNING] Unexpected: Ensemble loaded without errors!
This means either:
  1. All checkpoints have consistent architecture, OR
  2. Only 0-1 checkpoints were found (consistency check skipped)
   Training completed. Best F1: 0.8848
✓ Fold 3 retrained successfully

## Inspect Ensemble Properties

In [13]:
print(f"Ensemble Model Information:")
print(f"  Number of models: {len(ensemble.models)}")
print(f"  Device: {ensemble.device}")
print(f"\nModel Weights (normalized F1 scores):")
for i, weight in enumerate(ensemble.weights):
    print(f"  Fold {i}: {weight.item():.4f}")
print(f"\nWeights sum: {ensemble.weights.sum().item():.6f} (should be 1.0)")

Ensemble Model Information:
  Number of models: 4
  Device: cpu

Model Weights (normalized F1 scores):
  Fold 0: 0.2341
  Fold 1: 0.2552
  Fold 2: 0.2580
  Fold 3: 0.2528

Weights sum: 1.000000 (should be 1.0)


## Test Ensemble Inference

Create a small test batch and verify the ensemble produces predictions.

In [24]:
# Initialize dataloader if not already done
if pipeline.dataloader is None:
    from GradientGang.Pipeline.DataLoader.DataLoader import DataModule
    pipeline.dataloader = DataModule(params=pipeline.data_params)

# Setup data for inference
pipeline.dataloader.setup(stage="test", includeTestInTrain=False)

# Get a batch from test dataloader
test_batch, _ = next(iter(pipeline.dataloader.test_dataloader()))

# The batch is a list/tuple with [time_series_tensor, global_features_tensor]
time_series = test_batch[0]
global_features = test_batch[1]

print(f"Test batch shapes:")
print(f"  Time series: {time_series.shape}")
print(f"  Global features: {global_features.shape}")

# Run inference
ensemble.eval()
with torch.no_grad():
    # Forward pass (returns probabilities)
    probabilities = ensemble((time_series, global_features))
    # Get predictions
    predictions = torch.argmax(probabilities, dim=1)

print(f"\nEnsemble output:")
print(f"  Probabilities shape: {probabilities.shape}")
print(f"  Predictions shape: {predictions.shape}")
print(f"\nFirst 5 predictions: {predictions[:5].tolist()}")
print(f"First 5 probabilities (class 0): {probabilities[:5, 0].tolist()}")
print(f"First 5 probabilities (class 1): {probabilities[:5, 1].tolist()}")
print(f"First 5 probabilities (class 2): {probabilities[:5, 2].tolist()}")

Test batch shapes:
  Time series: torch.Size([32, 34, 160])
  Global features: torch.Size([32, 32])

Ensemble output:
  Probabilities shape: torch.Size([32, 3])
  Predictions shape: torch.Size([32])

First 5 predictions: [0, 0, 0, 0, 0]
First 5 probabilities (class 0): [0.9882528781890869, 0.9933563470840454, 0.9945709705352783, 0.9902544021606445, 0.9947105050086975]
First 5 probabilities (class 1): [0.00929438415914774, 0.0034315953962504864, 0.001493303570896387, 0.0025363084860146046, 0.0007957590278238058]
First 5 probabilities (class 2): [0.0024526817724108696, 0.0032119133975356817, 0.003935659304261208, 0.007209152448922396, 0.004493618384003639]


## Generate Submission with Ensemble

This will use the ensemble model to generate predictions for the test set.

In [ ]:
print("Generating submission with ensemble model...\n")

# This will automatically use load_ensemble_model() internally
submission_df = pipeline.create_submission()

Generating submission with ensemble model...

Loading K-fold ensemble...

LOADING K-FOLD ENSEMBLE
Best trial: 1
Best mean F1: 0.9138
Architecture: Autoencoder
------------------------------------------------------------
Best trial: 1
Best mean F1: 0.9138
Architecture: Autoencoder
------------------------------------------------------------
Reconstructing architecture...
✓ Architecture reconstructed

[STEP 1/3] Searching for checkpoints...
------------------------------------------------------------
  Fold 0: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_0/version_0/checkpoints\trial-1-fold-0-epoch=33-val_F1=0.904.ckpt
  Fold 1: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_1/version_0/checkpoints\trial-1-fold-1-epoch=40-val_F1=0.939.ckpt
  Fold 2: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_trial_1_fold_2/version_0/checkpoints\trial-1-fold-2-epoch=36-val_F1=0.921.ckpt
  Fold 3: Found @ FinalPipelineLogs/Study_TEST_TEST6/TEST_TEST6_tri

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...
   Training completed. Best F1: 0.8735
✓ Fold 0 retrained successfully: F1=0.8735
Validating fold 1 (expected F1: 0.9394)...
✓ Fold 1 validated: F1=0.8932
Validating fold 2 (expected F1: 0.9212)...
   Training completed. Best F1: 0.8735
✓ Fold 0 retrained successfully: F1=0.8735
Validating fold 1 (expected F1: 0.9394)...
✓ Fold 1 validated: F1=0.8932
Validating fold 2 (expected F1: 0.9212)...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


⚠️ Fold 2 degraded: F1=0.7962 < 0.8752
Fold 2: Validation failed - retraining...

🔄 Retraining fold 2...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


   Training completed. Best F1: 0.9273
✓ Fold 2 retrained successfully: F1=0.9273
Validating fold 3 (expected F1: 0.8909)...
⚠️ Fold 3 degraded: F1=0.7578 < 0.8464
Fold 3: Validation failed - retraining...

🔄 Retraining fold 3...
   MacroArchitecture: Autoencoder
   Use windowing: False
   Setting up fold data...
   Train batches: 16, Val batches: 6
   Starting training (max 100 epochs)...
   Training completed. Best F1: 0.8909
✓ Fold 3 retrained successfully: F1=0.8909
------------------------------------------------------------
[OK] All 4 models loaded/retrained
Fold F1 scores: ['0.8735', '0.8932', '0.9273', '0.8909']

✓ Ensemble created with performance-based weights
Weights: ['0.244', '0.249', '0.259', '0.249']
Device: cpu
   Training completed. Best F1: 0.8909
✓ Fold 3 retrained successfully: F1=0.8909
------------------------------------------------------------
[OK] All 4 models loaded/retrained
Fold F1 scores: ['0.8735', '0.8932', '0.9273', '0.8909']

✓ Ensemble created with per

TypeError: argument of type 'method' is not iterable

In [28]:
# Display first few rows
import pandas as pd
print(submission_path)
submission_df = submission_path
print(f"\nSubmission preview:")
print(submission_df.head(10))
print(f"\nSubmission shape: {submission_df.shape}")
print(f"Unique predictions: {submission_df['label'].unique()}")
print(f"Class distribution:")
print(submission_df['label'].value_counts())

     sample_index      label
0             000    no_pain
1             001    no_pain
2             002    no_pain
3             003    no_pain
4             004    no_pain
...           ...        ...
1319         1319    no_pain
1320         1320  high_pain
1321         1321    no_pain
1322         1322    no_pain
1323         1323    no_pain

[1324 rows x 2 columns]

Submission preview:
  sample_index    label
0          000  no_pain
1          001  no_pain
2          002  no_pain
3          003  no_pain
4          004  no_pain
5          005  no_pain
6          006  no_pain
7          007  no_pain
8          008  no_pain
9          009  no_pain

Submission shape: (1324, 2)
Unique predictions: ['no_pain' 'low_pain' 'high_pain']
Class distribution:
label
no_pain      973
high_pain    177
low_pain     174
Name: count, dtype: int64


## Test Summary

✅ **What was tested:**
1. Checkpoint search across multiple locations
2. Ensemble model loading with K-fold checkpoints
3. Weight calculation (normalized F1 scores)
4. Ensemble inference (weighted voting)
5. Submission generation with ensemble

⚠️ **If any folds were retrained:**
- Check the logs above for "Retraining fold X" messages
- Retraining is automatic and uses the same hyperparameters
- Retrained models are saved as checkpoints for future use

📊 **Expected behavior:**
- All K models should be loaded (or retrained if missing)
- Weights should sum to 1.0
- Higher performing folds should have higher weights
- Ensemble predictions should be stable and reproducible